
## Transform circuits data
1. Read bronze _`constructors`_ table
2. Keep only columns that are required (remove url col)
3. Standardize column names using snake_case(constructorId -> constructor_id)
4. Rename columns to make them more meaningful ( name -> constructors_name)
6. Remove duplicate rows
7. Transform values of nationality and to Title Case 
8. Write the tranformed data to silver table 

In [0]:
%run ../00-common/01.environment-config

In [0]:
%python
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"


**1. Read bronze races table**

In [0]:
# circuits_df = spark.read.option('VersionAsOf',0).table(bronze_table)
# use spark.read.table if you need additional options, otherwise use spark.table for simplicity

In [0]:
constructors_df = spark.table(bronze_table)

In [0]:
display(constructors_df)

**2. Keep only columns that are required (remove url col)**


In [0]:
# circuits_selected_df = circuits_df.select(
#     "circuitId",
#     "circuitName",
#     "lat",
#     "long",
#     "locality",
#     "country",
#     "Ingestion_timestamps",
#     "source_file"
# )

In [0]:
from pyspark.sql import functions as F

In [0]:
constructors_selected_df = constructors_df.select(
    F.col("constructorID"),
    F.col("name"),
    F.col("nationality"),
    F.col("Ingestion_timestamps"),
    F.col("source_file")
)

In [0]:
display(constructors_selected_df)

**Steps 3 and 4  Standardize column names**
- Standardize column names using snake_case 
- Rename columns to make them more meaningful 


In [0]:

# if you want to use withColumnRenamed it should be passed in indivdually .withColumnRenamed("circuitId","circuit_id") four times

In [0]:
constructors_renamed_df= (constructors_selected_df.withColumnsRenamed(
    {"constructorID": "constructor_id",
     "name": "constructor_name"
      })
    
    )

In [0]:
display(constructors_renamed_df)

In [0]:
# races_renamed_df.filter(F.col('season').isNull()).count()



**6. Remove duplicate rows**

In [0]:
#circuits_distinct_df = circuits_valid_df.distinct()

In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(['constructor_id'])

In [0]:
display(constructors_distinct_df)

**7. Transform values of _circuit_id_ and race_name to Title Case**


In [0]:
constructors_final_df = (
    constructors_distinct_df
    .withColumn("nationality", F.initcap(F.col("nationality")))

)

In [0]:
display(constructors_final_df)

**8. Write the tranformed data to silver table**

In [0]:
(constructors_final_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)
 
 

In [0]:
display(spark.table(silver_table))